# Units

Every physical setting of plasmacoll (a density, a temperature, a mass, an energy, a cross
section, a time step) takes an [astropy](https://www.astropy.org) Quantity, a string such as
`"300 K"` that astropy reads, or a plain number in the documented unit: SI, and eV for energies
and temperatures (`kT`). The value is converted once, when the setup is built, and stored as a
plain number, so the collision step never sees a Quantity. astropy is imported only when a
Quantity or a string is given; install it with `pip install plasmacoll[units]`.

In [ ]:
import astropy.units as u

import plasmacoll

gas = plasmacoll.NeutralBackground(
    "Ar",
    density=1e15 * u.cm**-3,
    temperature=300 * u.K,
    mass="39.948 u",
    drift=[0, 0, 1] * u.km / u.s,
)
gas

The temperature was given in K and is kept as `kT` in eV; K, eV and J convert into each other.
A plain number keeps its meaning, so `NeutralBackground("Ar", 1e21, 0.0259, 6.63e-26)` is the
same gas.

## Cross sections and processes

Tables take arrays with a unit, energies and thresholds any energy unit:

In [ ]:
ionization = plasmacoll.CrossSection(
    energy=[0.01576, 0.03, 0.1, 1.0] * u.keV,
    sigma="[0, 2.5, 2.8, 1.3] 1e-16 cm2",
    threshold="15.76 eV",
)
elastic = plasmacoll.CrossSection.constant("1e-15 cm2")
print(ionization.energy, ionization.sigma, ionization.threshold)

mcc = plasmacoll.MonteCarloCollisions(
    species_masses={"e": 1 * u.M_e, "Ar+": "39.947 u"},
    backgrounds=[gas],
    processes={
        "e": [
            plasmacoll.CollisionProcess("elastic", "Ar", elastic, energy_frame="lab"),
            plasmacoll.CollisionProcess(
                "ionization",
                "Ar",
                ionization,
                energy_frame="lab",
                products={"electron": "e", "ion": "Ar+"},
                energy_sharing="opal",
                sharing_energy=10 * u.eV,
            ),
        ]
    },
    seed=1,
)
mcc.species_masses

## A reactor in a field

The setup of `ZeroDReactor` and the time step of `run` take quantities too:

In [ ]:
electrons = plasmacoll.ParticleArrays.maxwellian(
    10_000, 1 * u.M_e, temperature="3 eV", weight=1e7, seed=2
)
reactor = plasmacoll.ZeroDReactor(
    mcc,
    {"e": electrons},
    volume=1 * u.cm**3,
    electric_field=[0, 0, 5] * u.kV / u.m,
    charges={"e": -1, "Ar+": 1},
)
history = reactor.run(dt="20 ps", num_steps=50, every=10)
print(f"{history.time[-1]:.3g} s, mean energy {history.mean_energy_ev['e'][-1]:.3g} eV")

The outputs are plain numbers in the documented units: seconds, eV, m^-3.

## Wrong units

A value of the wrong dimension raises:

In [ ]:
try:
    plasmacoll.NeutralBackground(
        "Ar", density=1 * u.Pa, temperature=300 * u.K, mass="40 u"
    )
except ValueError as error:
    print(error)

plasmacoll adds the Townsend to astropy: a reduced field for `solve_eedf` is `"10 Td"`,
`10 * plasmacoll.units.Td` or a plain number in Td.

The conversions live in `plasmacoll.units`, which other codes can use for their own settings:
`unit_field` and `convert_units` for dataclasses, `to_unit` and `to_array` for arguments, and
`with_unit` to write a value back to a configuration file:

## Units in your own setup

A code built on plasmacoll declares the unit of each setting of its dataclasses and converts them
once, in `__post_init__`. A field with a default takes `unit_field`; a field without one takes
`dataclasses.field(metadata=unit_metadata(...))`, which type checkers read as required. With
`array=True` an array stays an array (any backend) instead of becoming a tuple.


In [ ]:
import dataclasses

import numpy as np

from plasmacoll.units import convert_units, unit_field, unit_metadata


@dataclasses.dataclass(frozen=True)
class Discharge:
    density: np.ndarray = dataclasses.field(metadata=unit_metadata("m^-3", array=True))
    gap: float = dataclasses.field(metadata=unit_metadata("m"))
    reduced_field: float = unit_field("townsend", 100.0)

    def __post_init__(self) -> None:
        convert_units(self)


discharge = Discharge(
    np.linspace(1, 2, 3) * 1e11 * u.cm**-3, gap="5 mm", reduced_field="80 Td"
)
print(discharge.density, discharge.gap, discharge.reduced_field)

In [ ]:
from plasmacoll.units import to_unit, with_unit

print(to_unit("0.1 ns", "s"), with_unit(gas.density, "m^-3"))